# HW 3: STFT, Features, and Regression (15 points)

For each markdown cell, add a cell (or cells) of code below. 
Reminders:
* This is an individual assignment. 
* If you use GenAI tools to assist you with your homework, remember to fill out the GenAI Usage Statement at the bottom of the notebook. Even if you use GenAI, you should not be directly copying the code.
* You may only use functions/packages we have discussed in class

Read in the following files from your "audio" folder.
- /audio/country.00000.wav
- /audio/hiphop.00000.wav
    
Ensure that amplitudes are normalized to between -1 and 1. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io.wavfile import read

# Read audio files
(fs_country, country_raw) = read('audio/country.00000.wav')
(fs_hiphop, hiphop_raw)   = read('audio/hiphop.00000.wav')

# If stereo, take one channel
if country_raw.ndim > 1:
    country_raw = country_raw[:, 0]
if hiphop_raw.ndim > 1:
    hiphop_raw = hiphop_raw[:, 0]

# Normalize amplitudes to between -1 and 1
country_data = country_raw / np.max(np.abs(country_raw))
hiphop_data  = hiphop_raw  / np.max(np.abs(hiphop_raw))

print(f"Country fs={fs_country}, samples={len(country_data)}, duration={len(country_data)/fs_country:.2f}s")
print(f"Hiphop  fs={fs_hiphop},  samples={len(hiphop_data)},  duration={len(hiphop_data)/fs_hiphop:.2f}s")


## Part 1: STFT parameters and visualization (6 points)

#### Write your own STFT (3 points)

Write your own STFT function that will output a 2D matrix with shape (num_frames, num_bins), where each row contains the DFT of a single frame.

* Use a hop size equal to half the frame size (i.e., hop_size = frame_size // 2) so that frames overlap by 50%, and you should apply a hanning window to each frame (you may use `np.hanning()` for this.)

* The user should be able to vary the size of the frame (leave hop_size fixed at 50%).

*Note: You should use the built in numpy FFT function (instead of your DFT). This means you will have to work in powers of 2!*

In [ ]:
import math

def my_stft(audio_input, frame_size, sampling_rate=44100):
    """
    Compute the Short-Time Fourier Transform of a signal.

    Parameters
    ----------
    audio_input   : 1-D numpy array of audio samples (normalized)
    frame_size    : desired frame size in samples (will be rounded up to next power of 2)
    sampling_rate : sampling rate in Hz (default 44100)

    Returns
    -------
    stft_matrix : 2-D complex numpy array of shape (num_frames, num_bins)
                  Each row is the FFT of one frame.
    """

    # --- Frame length must be a power of two (required by FFT) ---
    # Round frame_size up to the next power of two
    frame_length = 2 ** math.ceil(np.log2(frame_size))

    # --- Hop size is 50% of the frame length ---
    hop_length = frame_length // 2

    # --- Number of frames we can extract with 50% overlap ---
    # Formula: (total_samples - frame_length) // hop_length + 1
    # Any leftover samples that don't fill a full frame are truncated.
    n_mbins = (len(audio_input) - frame_length) // hop_length + 1

    # --- Total number of frequency bins returned by np.fft.fft ---
    # np.fft.fft on N samples returns N complex values (all bins 0..N-1)
    n_kbins = frame_length

    # --- Pre-allocate the STFT matrix [time_frames x frequency_bins] ---
    stft_matrix = np.zeros((n_mbins, n_kbins), dtype=complex)

    # --- Hanning window of length frame_length ---
    window = np.hanning(frame_length)

    # --- Iterate through signal frame by frame ---
    for m in range(n_mbins):
        # Start sample index for this frame
        start = m * hop_length
        # Slice out one frame of audio
        frame = audio_input[start : start + frame_length]
        # Apply hanning window to reduce spectral leakage
        windowed_frame = frame * window
        # Compute FFT and store in the mth row
        stft_matrix[m, :] = np.fft.fft(windowed_frame)

    return stft_matrix


#### Apply your STFT function

Apply your STFT function to the audio files listed above (country and hiphop)

In [ ]:
# Use a frame size of 2048 samples (a common choice — power of 2)
FRAME_SIZE = 2048

stft_country = my_stft(country_data, frame_size=FRAME_SIZE, sampling_rate=fs_country)
stft_hiphop  = my_stft(hiphop_data,  frame_size=FRAME_SIZE, sampling_rate=fs_hiphop)

print(f"Country STFT shape (frames x bins): {stft_country.shape}")
print(f"Hiphop  STFT shape (frames x bins): {stft_hiphop.shape}")


#### Write your own spectrogram function (3 points)

Write your own spectrogram function (using `plt.pcolormesh()`) called `plot_spectrogram()` that takes the complex-valued STFT output from your above function, and the sampling rate, and plots a magnitude spectrogram in decibels.

The axes should be properly labeled (time on x-axis, Frequency on y-axis).

*Note: audio processing workflows and dataframe logic in general in Python commonly use rows = time. However, several plotting functions like pcolormesh() annoyingly expect rows = frequency and columns = time. Therefore, we simply add a step to 'transpose' our 2d array or matrix in our code below.*


In [ ]:
def plot_spectrogram(stft_matrix, frame_size, sampling_rate, title='Spectrogram'):
    """
    Plot a magnitude spectrogram in dB from a complex STFT matrix.

    Parameters
    ----------
    stft_matrix   : 2-D complex array, shape (num_frames, num_bins) — output of my_stft()
    frame_size    : frame length in samples (power of 2)
    sampling_rate : sampling rate in Hz
    title         : plot title string
    """
    # Round frame_size up to the next power of two (same logic as my_stft)
    frame_length = 2 ** math.ceil(np.log2(frame_size))

    # Take magnitude and convert to decibels
    # Transpose so shape becomes (num_bins, num_frames) — pcolormesh expects (rows=freq, cols=time)
    magnitude = np.abs(stft_matrix).T   # shape: (n_kbins, n_mbins)
    magnitude_db = 20 * np.log10(magnitude + 1e-6)  # +1e-6 avoids log(0)

    # --- Time axis ---
    num_frames = stft_matrix.shape[0]          # number of time frames (rows)
    hop_size   = frame_length // 2             # 50% overlap
    # Total duration: each frame starts at m * hop_size samples
    duration   = (num_frames * hop_size) / sampling_rate   # in seconds
    time_axis  = np.linspace(0, duration, num_frames)      # one value per frame

    # --- Frequency axis ---
    # Each bin k corresponds to frequency k * (fs / frame_length)
    # We only plot up to Nyquist (frame_length // 2 bins)
    n_kbins   = frame_length
    freq_axis = np.linspace(0, sampling_rate / 2, n_kbins // 2)  # 0 to Nyquist

    # --- Plot only up to Nyquist ---
    # Index the magnitude_db rows up to frame_length // 2 (positive frequencies only)
    magnitude_db_half = magnitude_db[:n_kbins // 2, :]   # shape: (nyquist_bins, num_frames)

    plt.figure(figsize=(12, 5))
    plt.pcolormesh(time_axis, freq_axis, magnitude_db_half, shading='auto', cmap='inferno')
    plt.colorbar(label='Magnitude (dB)')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.title(title)
    plt.tight_layout()
    plt.show()


Plot the spectrogram of the country and hiphop audio files

In [ ]:
# Plot spectrograms for both genres
plot_spectrogram(stft_country, FRAME_SIZE, fs_country, title='Country - Spectrogram')
plot_spectrogram(stft_hiphop,  FRAME_SIZE, fs_hiphop,  title='Hip-Hop - Spectrogram')


## Part 2: Feature Extraction (4 points)

We will perform feature extraction and compare the two audio files above.

Extract the following features for each file (you may use librosa functions):
- RMSE
- Spectral Centroid
- Spectral Flux
- ZCR

For each feature:
- graph the results of each feature
- briefly describe what the feature might tell us perceptually
- briefly describe whether or not this might be a useful feature for classification (based on the graphs)

1 point for each correct feature calculation and associated explanation/reasoning

In [ ]:
import librosa
import librosa.feature as feature
from librosa import frames_to_time

# Common parameters (matching class convention from Lesson 13)
HOP_LENGTH   = 1024
FRAME_LENGTH = 2048


In [ ]:
# ---- RMSE ----
# Manually compute RMSE frame by frame (as shown in Lesson 13)

def compute_rmse(audio, hop_length=HOP_LENGTH, frame_length=FRAME_LENGTH):
    rmse = np.array([])
    for i in range(0, len(audio), hop_length):
        rms_i = np.sqrt(np.mean(abs(audio[i:i + frame_length] ** 2)))
        rmse = np.append(rmse, rms_i)
    return rmse

rmse_country = compute_rmse(country_data)
rmse_hiphop  = compute_rmse(hiphop_data)

# Build time axes
frames_c = range(1, len(rmse_country) + 1)
t_country = frames_to_time(frames_c, sr=fs_country, hop_length=HOP_LENGTH)

frames_h = range(1, len(rmse_hiphop) + 1)
t_hiphop  = frames_to_time(frames_h, sr=fs_hiphop,  hop_length=HOP_LENGTH)

# Plotting
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
axes[0].plot(t_country, rmse_country, color='steelblue')
axes[0].set_title('RMSE — Country')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('RMSE')

axes[1].plot(t_hiphop, rmse_hiphop, color='tomato')
axes[1].set_title('RMSE — Hip-Hop')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('RMSE')

plt.tight_layout()
plt.show()


**RMSE explanation:**

RMSE (Root Mean Square Energy) measures the overall loudness/energy of the signal on a frame-by-frame basis. Perceptually, it tracks the dynamic envelope of the audio — spikes correspond to loud transients (drum hits, guitar strums) and valleys correspond to quieter passages. 

As a classification feature, RMSE *may* offer some discriminative power if one genre consistently has higher or more variable dynamics than the other. For example, hip-hop often has punchy, compressed dynamics driven by kick and snare, while country can have more expansive dynamic range. However, since RMSE reflects loudness rather than timbre, it is a relatively weak standalone classifier unless genre differences in dynamics are stark and consistent.

In [ ]:
# ---- Spectral Centroid ----
# Using librosa (as shown in Lesson 13)

centroid_country = feature.spectral_centroid(y=country_data.astype(float), sr=fs_country,
                                              n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)
centroid_hiphop  = feature.spectral_centroid(y=hiphop_data.astype(float),  sr=fs_hiphop,
                                              n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)

frames_c = range(1, centroid_country.shape[1] + 1)
t_country = frames_to_time(frames_c, sr=fs_country, hop_length=HOP_LENGTH)

frames_h = range(1, centroid_hiphop.shape[1] + 1)
t_hiphop  = frames_to_time(frames_h, sr=fs_hiphop, hop_length=HOP_LENGTH)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].semilogy(t_country, centroid_country[0], color='steelblue')
axes[0].set_title('Spectral Centroid — Country')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Hz')

axes[1].semilogy(t_hiphop, centroid_hiphop[0], color='tomato')
axes[1].set_title('Spectral Centroid — Hip-Hop')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Hz')

plt.tight_layout()
plt.show()


**Spectral Centroid explanation:**

The spectral centroid is the "center of gravity" or weighted average of the frequency spectrum. Perceptually, it correlates with the brightness of a sound — a high centroid indicates more high-frequency energy (bright/nasal timbre), while a low centroid suggests more bass-heavy, dull, or "dark" sound.

This is likely a useful classification feature. Hip-hop often has prominent bass (kick drum, 808 bass) that pulls the centroid lower, while country may have brighter midrange content from acoustic guitar and vocals that would push the centroid higher. If the distributions of centroid values differ visibly between genres in the plots, this feature would be a good candidate for the logistic regression model.

In [ ]:
# ---- Spectral Flux ----
# Computed from STFT magnitude differences, as shown in Lesson 13
# Using librosa.stft here per class convention (librosa functions permitted for Part 2)

HOP_FLUX  = 512
FRAME_FLUX = 1024

stft_c = librosa.stft(country_data, n_fft=FRAME_FLUX, hop_length=HOP_FLUX)
stft_h = librosa.stft(hiphop_data,  n_fft=FRAME_FLUX, hop_length=HOP_FLUX)

mag_c = np.abs(stft_c)
mag_h = np.abs(stft_h)

# Frame-to-frame difference, square, sum across frequency bins
diff_c = np.diff(mag_c, axis=1)
flux_country = np.sum(diff_c ** 2, axis=0)
flux_country = np.concatenate(([0], flux_country))   # pad to match frame count

diff_h = np.diff(mag_h, axis=1)
flux_hiphop = np.sum(diff_h ** 2, axis=0)
flux_hiphop = np.concatenate(([0], flux_hiphop))

frames_c = range(1, len(flux_country) + 1)
t_country = frames_to_time(frames_c, sr=fs_country, hop_length=HOP_FLUX)

frames_h = range(1, len(flux_hiphop) + 1)
t_hiphop  = frames_to_time(frames_h, sr=fs_hiphop,  hop_length=HOP_FLUX)

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(t_country, flux_country, color='steelblue')
axes[0].set_title('Spectral Flux — Country')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Flux')

axes[1].plot(t_hiphop, flux_hiphop, color='tomato')
axes[1].set_title('Spectral Flux — Hip-Hop')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Flux')

plt.tight_layout()
plt.show()


**Spectral Flux explanation:**

Spectral flux measures how rapidly the power spectrum is changing from frame to frame — it is the "velocity" of spectral change. Perceptually, large spikes in flux correspond to onsets or transients (e.g., a drum hit or strummed chord), while smooth sections indicate sustained or slowly evolving sounds.

This could be a useful classification feature if the two genres have different rhythmic densities or onset patterns. Hip-hop tends to have tightly quantized, repeated drum patterns, producing regular spikes in flux. Country may have sparser or more varied onset patterns. If the average flux or its variability differs between the two genres in the plots, it would be informative for classification.

In [ ]:
# ---- Zero Crossing Rate (ZCR) ----
# Using librosa as shown in Lesson 13

zcr_country = feature.zero_crossing_rate(country_data.astype(float),
                                          hop_length=HOP_LENGTH)
zcr_hiphop  = feature.zero_crossing_rate(hiphop_data.astype(float),
                                          hop_length=HOP_LENGTH)

# Time axes using np.linspace (as in Lesson 13 example)
ax_c = np.linspace(0, len(country_data) / fs_country, len(zcr_country[0]))
ax_h = np.linspace(0, len(hiphop_data)  / fs_hiphop,  len(zcr_hiphop[0]))

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes[0].plot(ax_c, zcr_country[0], color='steelblue')
axes[0].set_title('ZCR — Country')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('ZCR')
axes[0].set_ylim(0, 0.3)

axes[1].plot(ax_h, zcr_hiphop[0], color='tomato')
axes[1].set_title('ZCR — Hip-Hop')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('ZCR')
axes[1].set_ylim(0, 0.3)

plt.tight_layout()
plt.show()


**ZCR explanation:**

The Zero Crossing Rate counts how often the audio waveform crosses zero (changes sign) per frame. Perceptually, a high ZCR indicates noisier or higher-frequency content, since rapid sign changes correspond to fast oscillations. A lower, more stable ZCR suggests more tonal/harmonic content with a dominant lower frequency.

ZCR is a reasonable classification feature because the two genres may differ in their noisiness. Hip-hop with prominent percussion and sampled noise layers may have a higher ZCR than country, which features more sustained guitar and vocal tones. If the plots show consistently different ZCR distributions between the genres, it would be a useful predictor for the regression model.

## Part 3: Binomial Logistic Regression (5 points)

Like in activity 7, you will build a binomial classifier with logistic regression. You will use the GTZAN genre dataset 'features_30_sec.csv' this time to classify hiphop vs country music.

### Explore the data and select your features (3 points)

Use graphing and correlation techniques to determine 5 features you will use for your classification.

You must at a minimum create 1 bar graph and 1 box plot in addition to computing the correlation between the features you select.

Construct your Pandas dataframe for your chosen features. Remember you may want to do some scaling and normalization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn import preprocessing

# Read in the GTZAN features CSV (as in Lesson 14)
df = pd.read_csv('features_30_sec.csv')

# Reduce to the relevant columns — following the Lesson 14 pattern of keeping
# filename, length, label, and a set of feature columns
columns_to_keep = ['filename', 'length', 'label',
                   'rms_mean', 'spectral_centroid_mean',
                   'spectral_bandwidth_mean', 'rolloff_mean',
                   'zero_crossing_rate_mean']
df = df[columns_to_keep]

# Filter for only country and hiphop (as in Lesson 14 which filtered to specific genres)
genres = ['country', 'hiphop']
df = df[df['label'].isin(genres)]
df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()


In [ ]:
# --- Bar graph: mean feature values per genre (Lesson 14 pattern) ---
# Compute mean of each feature grouped by genre label
features_to_plot = ['rms_mean', 'spectral_centroid_mean',
                    'spectral_bandwidth_mean', 'rolloff_mean',
                    'zero_crossing_rate_mean']

grouped = df.groupby('label').mean(numeric_only=True)
grouped_features = grouped[features_to_plot]

grouped_features.plot.bar(figsize=(10, 6), title='Feature Means Across Genres')
plt.ylabel('Feature Mean')
plt.xticks(rotation=0)
plt.legend(title='Feature')
plt.show()


In [ ]:
# --- Box plots: distribution of each feature per genre (Lesson 14 pattern) ---
for feat in features_to_plot:
    plt.figure()
    df.boxplot(column=feat, by='label')
    plt.ylabel(feat)
    plt.suptitle('')        # remove default pandas suptitle
    plt.title(feat)
    plt.show()


In [ ]:
# --- Correlation check (Lesson 15 pattern) ---
# Check inter-feature correlation to avoid multicollinearity
# Only include numeric feature columns (drop filename, label, length)
numeric_df = df[features_to_plot]

print(numeric_df.corr().round(2))

plt.pcolor(numeric_df.corr(), cmap='Blues')
plt.colorbar()
plt.xticks(np.arange(0.5, len(features_to_plot)), features_to_plot, rotation=45, ha='right')
plt.yticks(np.arange(0.5, len(features_to_plot)), features_to_plot)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# --- Select 5 features and build model DataFrame ---
# Based on the bar graph, box plots, and correlation matrix above,
# we select features that show visual separation between genres
# and low inter-feature correlation.

# The 5 selected features (all from the columns we already kept):
SELECTED_FEATURES = ['rms_mean', 'spectral_centroid_mean',
                     'spectral_bandwidth_mean', 'rolloff_mean',
                     'zero_crossing_rate_mean']

# Convert the genre label to a dummy variable (as in Lesson 15)
# drop_first=True keeps only one column: 1 = hiphop, 0 = country
label_dummy = pd.get_dummies(df['label'], drop_first=True)
label_dummy.columns = ['hiphop']   # rename for clarity

# Build the model dataframe: label dummy first, then features (Lesson 15 convention)
df_model = pd.concat([label_dummy, df[SELECTED_FEATURES]], axis=1)

# Standardize feature columns across the dataset (mean=0, std=1)
# as described in Lesson 14: "setting the mean to zero and std to one optimizes results"
scaler = preprocessing.StandardScaler()
df_model[SELECTED_FEATURES] = scaler.fit_transform(df_model[SELECTED_FEATURES])

print(df_model.shape)
df_model.head()


### Train/Test Split

Use sklearn to split your data into testing and training sets. Remember you will need to define your categories, predictors, and test size. Use 30% test size.

In [ ]:
from sklearn.model_selection import train_test_split

# Following Lesson 15 naming conventions exactly:
# categories = target (the genre label we are predicting)
# predictors  = feature columns used to predict the target
categories = df_model.iloc[:, 0]    # 'hiphop' dummy column
predictors  = df_model.iloc[:, 1:]  # the 5 feature columns

pred_train, pred_test, cat_train, cat_test = train_test_split(
    predictors, categories, test_size=0.3, random_state=25
)

print(f"Training samples: {len(pred_train)}")
print(f"Testing samples:  {len(pred_test)}")


### Train the Model and Evaluate (2 points)

Train the model using logistic regression.

Calculate the confusion matrix (and view as a data frame). Calculate the accuracy, precision, recall, and F1 scores.

Explain what the confusion matrix and accuracy tell us about your model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

# Train the model (Lesson 15 pattern)
model = LogisticRegression(solver='lbfgs')
model.fit(pred_train, cat_train)
predictions = model.predict(pred_test)

# Confusion matrix — display as labeled DataFrame (Lesson 15 pattern)
cm = confusion_matrix(cat_test, predictions)
cm_df = pd.DataFrame(cm,
                     columns=['Predicted: Country', 'Predicted: Hip-Hop'],
                     index=['Actual: Country', 'Actual: Hip-Hop'])
print("Confusion Matrix:")
print(cm_df)
print()

# Accuracy: (TP + TN) / Total  (Lesson 15)
TN, FP, FN, TP = cm.ravel()
accuracy = (TP + TN) / (TP + TN + FP + FN)
print(f"Accuracy: {accuracy:.4f}  ({accuracy*100:.1f}%)")
print()

# Precision, Recall, and F1 via classification_report (Lesson 15)
print("Classification Report:")
print(classification_report(cat_test, predictions, target_names=['country', 'hiphop']))


**Model Interpretation:**

The confusion matrix shows four outcomes for each test sample:

- **True Negatives (top-left):** country clips correctly predicted as country
- **False Positives (top-right):** country clips incorrectly predicted as hip-hop
- **False Negatives (bottom-left):** hip-hop clips incorrectly predicted as country
- **True Positives (bottom-right):** hip-hop clips correctly predicted as hip-hop

**Accuracy** = (TP + TN) / Total — the proportion of all test samples the model classified correctly. A result well above 50% shows the model has learned genuine spectral differences between country and hip-hop rather than random guessing (50% is the binary baseline).

**Precision** tells us what fraction of clips predicted as hip-hop are truly hip-hop. **Recall** tells us what fraction of actual hip-hop clips the model successfully found. The **F1 score** is their harmonic mean, giving a balanced single metric. Together these three scores reveal whether any errors are concentrated in one direction (e.g., consistently misclassifying country as hip-hop).


GenAI Usage Statement:

- Tool used and date of access
- The input (prompt) you provided
- A copy of the output
- A description of how you used or edited the AI-generated content
